<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/NB03_2_taxon_directed_BLASTsearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taxon-Directed Homolog Search  (consumer)

Searches a **single protein query** against the shared proteome library and keeps the best homologs
per organism. It only **reads** the library — proteomes and BLAST databases are made by the
`Proteome Library Builder` notebook (run that first).

### How to use

1. **Once:** put your query FASTA files in the `queries/` folder (Section 0 creates it).
2. **Each run:** in Section 1 set `QUERY_FASTA` to the file you want — that is the only required edit —
   then **Runtime ▸ Run all**. Results appear in `Outputs/<COLLECTION>/<QUERY_NAME>/<RUN_LABEL>/`.
3. **Advanced — same query, different parameters:** change `EVALUE` / `TOP_N` / `MIN_QUERY_COVERAGE`
   **and** give `RUN_LABEL` a new name, then Run all again. The new results go to a separate folder and
   the earlier run is kept, so you can compare parameter settings side by side.

## 0. Setup — mount Drive and create folders

Sets `PROJECT_DRIVE` and `COLLECTION` (the only place). Creates a **persistent query folder**,
`Data/<COLLECTION>/queries/` — keep all your query FASTAs there; drop them in once and the notebook
reads whichever one you name. The shared library must already exist in `Databases/proteome_library/`
(built by the Library Builder); Section 3 checks that.

In [61]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

# ---- LOCATION (the only place these are set) ----
PROJECT_DRIVE = "/content/drive/MyDrive/BIOINFO4-5203-F26"   # your project folder on Drive
COLLECTION    = "NB03_taxon_directed_homologs"              # supervisory folder for this notebook's runs

PROJECT     = Path(PROJECT_DRIVE)
QUERIES_DIR = PROJECT / "Data" / COLLECTION / "queries"     # PERSISTENT: keep ALL your query FASTAs here
for d in (QUERIES_DIR, PROJECT / "Outputs" / COLLECTION, PROJECT / "Notebooks", PROJECT / "Reports"):
    d.mkdir(parents=True, exist_ok=True)
print("project:", PROJECT)
print("Put your query FASTA files (once) in:", QUERIES_DIR)
print("Reads the shared library from Databases/proteome_library/ (built by the Library Builder).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project: /content/drive/MyDrive/BIOINFO4-5203-F26
Put your query FASTA files (once) in: /content/drive/MyDrive/BIOINFO4-5203-F26/Data/NB03_taxon_directed_homologs/queries
Reads the shared library from Databases/proteome_library/ (built by the Library Builder).


## 1. Configuration

Set the query and, if you like, the search parameters. `QUERY_FASTA` is just the **filename** of a
FASTA already in your `queries/` folder — the only edit a routine run needs. `QUERY_NAME` labels the
output folders; leave it **blank** to name it automatically from the file, or set it for a custom
label. `RUN_LABEL` names the results subfolder for **this parameter set**, so the same query at
different parameters is saved side by side. **KEY SEARCH PARAMETERS** tune the search itself.

In [67]:
import subprocess, sys, os
# dependencies: Biopython (read sequences) + BLAST+ (blastp). No downloader needed — the library exists.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython==1.85", "pandas"], check=True)
if subprocess.run("which blastp", shell=True, capture_output=True).returncode != 0:
    subprocess.run("apt-get -qq update && apt-get -qq install -y ncbi-blast+", shell=True, check=True)
from Bio import Entrez

# ============================================================================
# INPUTS
# ============================================================================
LIBRARY_CSV = "Taxon_library_final.csv"   # taxon table (shared) in Databases/proteome_library/
QUERY_FASTA = "Cyt-c_cyano.faa"           # filename of a query FASTA already in your queries/ folder
QUERY_NAME  = ""                           # label for the output folders. BLANK = auto (filename without extension)
RUN_LABEL   = "E4_cov50"                   # names the RESULTS subfolder for THIS parameter set
Entrez.email = "rob.burnap@okstate.edu"    # NCBI requires a contact email

# ============================================================================
# KEY SEARCH PARAMETERS  —  the science knobs you will actually tune
# ============================================================================
EVALUE = 1e-4               # BLAST significance cutoff. LOWER = stricter (fewer, more confident hits).
                           #   Try 1e-20 for close homologs only; 1e-3 to reach divergent ones.
TOP_N  = 1                 # how many homologs to KEEP per organism. >1 preserves PARALOGS
                           #   instead of collapsing to a single best hit.
MIN_QUERY_COVERAGE = 0    # minimum % of the QUERY length the alignment must span (best HSP).
                           #   0 = off. Raise to 50-70 to reject short / partial-domain hits.

# ============================================================================
# ADVANCED
# ============================================================================
TAXID_COLUMN = "auto"      # which library column holds the TaxID (auto = best available)

## 2. Prepare the run — pick the query and create the results folder

Confirms `QUERY_FASTA` is actually in your `queries/` folder (and, if the name doesn't match, lists
what *is* there), checks it is a single sequence, sets `QUERY_NAME` (from the filename if you left it
blank), and creates this run's results folder. If the name is wrong or the file hasn't been dropped in
yet, this cell **stops with a clear message** — nothing runs on a missing query.

In [68]:
from Bio import SeqIO

# 1) resolve the query: an absolute path, else a file in the persistent queries/ folder
qp = Path(QUERY_FASTA)
QUERY_PATH = qp if qp.is_absolute() else QUERIES_DIR / QUERY_FASTA

# 2) if it is not there, show what IS available and stop
if not QUERY_PATH.exists():
    available = sorted(p.name for p in QUERIES_DIR.glob("*.f*a")) or ["(none yet - drop a FASTA in)"]
    raise FileNotFoundError(
        f"Query '{QUERY_FASTA}' not found in {QUERIES_DIR}\n"
        f"Available query files there:\n  " + "\n  ".join(available) +
        f"\nSet QUERY_FASTA (Section 1) to one of these, or drop the file into that folder.")

# 3) confirm it is a single sequence and show its ID
recs = list(SeqIO.parse(QUERY_PATH, "fasta"))
if not recs:
    raise ValueError(f"{QUERY_PATH} contains no sequences.")
if len(recs) > 1:
    print(f"NOTE: {len(recs)} sequences in {QUERY_FASTA}; the search uses the first ({recs[0].id}).")

# 4) label for the output folders: auto from the filename unless QUERY_NAME was set
QUERY_NAME = QUERY_NAME.strip() or Path(QUERY_FASTA).stem

# 5) create this run's results folder (keyed by QUERY_NAME + RUN_LABEL)
OUT_ROOT  = PROJECT / "Outputs" / COLLECTION / QUERY_NAME / RUN_LABEL
RUNS_DIR  = OUT_ROOT
STATE_DIR = OUT_ROOT / "state"
for d in (OUT_ROOT, STATE_DIR):
    d.mkdir(parents=True, exist_ok=True)
EVENTS      = RUNS_DIR / "events.jsonl"
SUMMARY_CSV = RUNS_DIR / f"{QUERY_NAME}_{RUN_LABEL}_summary.csv"
HITS_FASTA  = RUNS_DIR / f"{QUERY_NAME}_{RUN_LABEL}_hits.faa"

print("RUN SUMMARY")
print(f"  query      : {QUERY_FASTA}   (sequence id: {recs[0].id})")
print(f"  name/label : {QUERY_NAME}  /  {RUN_LABEL}")
print(f"  parameters : EVALUE={EVALUE}   TOP_N={TOP_N}   MIN_QUERY_COVERAGE={MIN_QUERY_COVERAGE}")
print(f"  library    : {LIBRARY_CSV}")
print(f"  results -> {OUT_ROOT}")

# 6) warn if this RUN_LABEL already holds results from a DIFFERENT parameter set
import json as _json
_mf = OUT_ROOT / "manifest.json"
if _mf.exists():
    prev = _json.loads(_mf.read_text())
    if (prev.get("evalue"), prev.get("min_coverage"), prev.get("top_n")) == (EVALUE, MIN_QUERY_COVERAGE, TOP_N):
        print(f"\n  NOTE: run '{RUN_LABEL}' already exists with these parameters; the search will extend it.")
    else:
        print(f"\n  *** WARNING: run '{RUN_LABEL}' already has results from DIFFERENT parameters "
              f"(EVALUE={prev.get('evalue')} TOP_N={prev.get('top_n')} COV={prev.get('min_coverage')}).")
        print("      Running the search REPLACES them. To keep them, give RUN_LABEL a new name (Section 1).")

print("\nReady. Run Sections 3-5 (validate, helpers, search).")

RUN SUMMARY
  query      : Cyt-c_cyano.faa   (sequence id: cyt-c6)
  name/label : Cyt-c_cyano  /  E4_cov50
  parameters : EVALUE=0.0001   TOP_N=1   MIN_QUERY_COVERAGE=0
  library    : Taxon_library_final.csv
  results -> /content/drive/MyDrive/BIOINFO4-5203-F26/Outputs/NB03_taxon_directed_homologs/Cyt-c_cyano/E4_cov50

Ready. Run Sections 3-5 (validate, helpers, search).


## 3. Validate and locate the library

Confirms the shared library has been built (by the Proteome Library Builder), then loads the taxon
table — used only for organism labels and ordering. The per-run folders were created in Section 2.

In [69]:
import pandas as pd, json, glob

DB_ROOT        = PROJECT / "Databases" / "proteome_library"   # SHARED library (read-only here)
PROTEOME_DIR   = DB_ROOT / "proteomes"                         # full hit sequences are read from here
DB_DIR         = DB_ROOT / "blastdbs"                          # BLAST databases to search
LIB_STATUS_CSV = DB_ROOT / "manifest" / "library_status.csv"  # which taxa are ready (from the builder)

if "@" not in Entrez.email:
    raise ValueError("Set Entrez.email in Section 1.")

# the library must already be built by the Proteome Library Builder notebook
if not glob.glob(str(DB_DIR / "*.p*")):
    raise FileNotFoundError(f"No BLAST databases found in {DB_DIR}.\n"
                            f"Run the Proteome Library Builder notebook first.")

lib_path = DB_ROOT / LIBRARY_CSV
if not lib_path.exists():
    raise FileNotFoundError(f"Taxon table not found at {lib_path}.")
library = pd.read_csv(lib_path)

def pick_taxid_col(df):
    if TAXID_COLUMN != "auto" and TAXID_COLUMN in df.columns:
        return TAXID_COLUMN
    for c in ("resolved_taxid", "NCBI_TaxID", "taxid", "species_taxid"):
        if c in df.columns:
            return c
    raise ValueError(f"No TaxID column in {list(df.columns)}")
TAXCOL = pick_taxid_col(library)

def _as_taxid(v):
    try:    return int(float(v))
    except (TypeError, ValueError): return None

targets = []
for _, r in library.iterrows():
    tx = _as_taxid(r.get(TAXCOL))
    if tx is None:
        continue
    targets.append({"taxid": tx, "label": str(r.get("Organism", f"taxid_{tx}")).strip(),
                    "tier": r.get("Tier", "")})

print(f"library: {len(targets)} taxa in table; {len(glob.glob(str(DB_DIR/'*.p*')))} database files present.")

library: 197 taxa in table; 1358 database files present.


## 4. Search helpers

The reusable functions the search runs on. `run_blastp` applies `EVALUE`; `select_top_hits` applies
`MIN_QUERY_COVERAGE` and keeps the best `TOP_N`; `attach_sequences` reads each hit's full sequence
from the local proteome (no network). Checkpoint/export helpers let a rerun resume.

In [70]:
import json, glob, os, subprocess, hashlib
from datetime import datetime, timezone
from Bio import SeqIO

def utc_now(): return datetime.now(timezone.utc).isoformat()
def atomic_text(path, text):
    tmp = path.with_name(path.name + ".tmp"); tmp.write_text(text, encoding="utf-8"); tmp.replace(path)
def save_json(path, data): atomic_text(path, json.dumps(data, indent=2, ensure_ascii=False) + "\n")
def event(kind, **d):
    with EVENTS.open("a", encoding="utf-8") as h:
        h.write(json.dumps({"time": utc_now(), "kind": kind, **d}) + "\n")
def run_cmd(cmd, timeout=1800):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return p.returncode, p.stdout, p.stderr

def proteome_path(taxid): return PROTEOME_DIR / f"{taxid}.faa"
def db_prefix(taxid):     return DB_DIR / str(taxid)
def db_ready(taxid):      return bool(glob.glob(str(db_prefix(taxid)) + ".p*"))

# BLAST tabular output (outfmt 6) columns we request, in order:
FIELDS6 = "qseqid sseqid pident length qstart qend sstart send evalue bitscore qlen slen stitle"

def run_blastp(query_fasta, taxid):
    """BLAST the query against ONE organism's local database. EVALUE is applied here by BLAST."""
    rc, out, err = run_cmd(["blastp", "-query", str(query_fasta), "-db", str(db_prefix(taxid)),
                            "-outfmt", "6 " + FIELDS6, "-evalue", str(EVALUE),
                            "-max_target_seqs", "50", "-num_threads", "2"])
    if rc != 0:
        raise RuntimeError(f"blastp failed for {taxid}: {err[:300]}")
    return out                                       # raw tab-separated hit lines (may be empty)

def select_top_hits(blast_out):
    """(1) best HSP per subject protein; (2) drop hits below MIN_QUERY_COVERAGE / above EVALUE;
    (3) rank by bitscore and keep the top TOP_N — so paralogs survive, not just the best."""
    best = {}                                        # subject protein id -> its best-scoring HSP
    for line in blast_out.splitlines():
        f = line.split("\t")
        if len(f) < 13:
            continue
        sseqid = f[1]; evalue = float(f[8]); bits = float(f[9])
        qs, qe, qlen = int(f[4]), int(f[5]), int(f[10])
        cov = 100.0 * (abs(qe - qs) + 1) / max(1, qlen)   # % of the QUERY covered by this HSP
        rec = {"sseqid": sseqid, "pident": float(f[2]), "align_length": int(f[3]),
               "evalue": evalue, "bitscore": bits, "query_coverage_pct": cov,
               "slen": int(f[11]), "stitle": f[12]}
        if sseqid not in best or bits > best[sseqid]["bitscore"]:
            best[sseqid] = rec
    kept = [r for r in best.values() if r["evalue"] <= EVALUE and r["query_coverage_pct"] >= MIN_QUERY_COVERAGE]
    kept.sort(key=lambda r: (-r["bitscore"], r["evalue"]))
    return kept[:TOP_N]

def attach_sequences(taxid, hits):
    """Read each hit's full protein sequence from the local proteome (no network)."""
    if not hits:
        return hits
    idx = SeqIO.index(str(proteome_path(taxid)), "fasta")
    try:
        for h in hits:
            rec = idx.get(h["sseqid"])
            h["sequence"] = str(rec.seq) if rec is not None else ""
    finally:
        idx.close()
    return hits

def checkpoint_path(taxid): return STATE_DIR / f"{taxid}.result.json"
def load_checkpoints():
    done = {}
    for t in targets:
        p = checkpoint_path(t["taxid"])
        if p.exists():
            done[t["taxid"]] = json.loads(p.read_text(encoding="utf-8"))
    return done

def export_results(completed):
    """Rewrite the summary CSV and hits FASTA from all completed taxa (atomic, safe to call often)."""
    import csv, io
    fields = ["query", "taxid", "label", "tier", "rank", "sseqid", "pident", "evalue", "bitscore",
              "query_coverage_pct", "align_length", "slen", "stitle", "fasta_id"]
    buf = io.StringIO(newline=""); w = csv.DictWriter(buf, fieldnames=fields); w.writeheader()
    fasta = []
    for t in targets:
        item = completed.get(t["taxid"])
        if not item:
            continue
        for rank, h in enumerate(item.get("hits", []), 1):
            fid = f"{h['sseqid']}__txid{t['taxid']}__{QUERY_NAME}__h{rank}"
            w.writerow({"query": QUERY_NAME, "taxid": t["taxid"], "label": t["label"], "tier": t["tier"],
                        "rank": rank, "sseqid": h["sseqid"], "pident": round(h["pident"], 1),
                        "evalue": h["evalue"], "bitscore": h["bitscore"],
                        "query_coverage_pct": round(h["query_coverage_pct"], 1),
                        "align_length": h["align_length"], "slen": h["slen"],
                        "stitle": h.get("stitle", ""), "fasta_id": fid})
            seq = h.get("sequence", "")
            if seq:
                wrapped = "\n".join(seq[i:i+80] for i in range(0, len(seq), 80))
                fasta.append(f">{fid} {t['label']} | {h.get('stitle','')}\n{wrapped}\n")
    atomic_text(SUMMARY_CSV, buf.getvalue())
    atomic_text(HITS_FASTA, "".join(fasta))


## 5. Search the library

Runs the query against every ready organism and keeps the top `TOP_N` hits within threshold.
Checkpointed per organism; changing the query or any search parameter automatically clears old
checkpoints and re-searches. Outputs `<QUERY_NAME>_summary.csv` and `<QUERY_NAME>_hits.faa`.

In [71]:
# the query was resolved and validated in Section 2
qpath = QUERY_PATH
qrecs = list(SeqIO.parse(qpath, "fasta"))
if not qrecs:
    raise ValueError(f"No sequence in {qpath}")
if len(qrecs) > 1:
    print(f"NOTE: {len(qrecs)} sequences in query; using the first ({qrecs[0].id}).")
query_seq = str(qrecs[0].seq)

# signature guard: changing the query or any threshold invalidates old checkpoints for this run
sig = hashlib.sha256(("|".join([QUERY_NAME, query_seq, str(EVALUE),
                                str(MIN_QUERY_COVERAGE), str(TOP_N)])).encode()).hexdigest()
manifest_path = RUNS_DIR / "manifest.json"
old = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
if old.get("signature") != sig:
    for f in STATE_DIR.glob("*.result.json"):
        f.unlink()
    save_json(manifest_path, {"signature": sig, "query": QUERY_NAME, "created": utc_now(),
                              "evalue": EVALUE, "min_coverage": MIN_QUERY_COVERAGE, "top_n": TOP_N})
    print("New/changed query or thresholds - previous checkpoints for this query cleared.")

# search only the organisms the builder marked ready (NaN/empty error = success)
st = pd.read_csv(LIB_STATUS_CSV)
ready_ids = set(int(x) for x in st[st["error"].isna() | (st["error"].astype(str).str.strip() == "")]["taxid"])
ready = [t for t in targets if t["taxid"] in ready_ids and db_ready(t["taxid"])]

completed = load_checkpoints()
export_results(completed)
pending = [t for t in ready if t["taxid"] not in completed]
print(f"{len(completed)}/{len(ready)} taxa already searched for {QUERY_NAME}; {len(pending)} to go.")

for i, t in enumerate(pending, 1):
    taxid = t["taxid"]
    print(f"[{i}/{len(pending)}] {t['label']} (TaxID {taxid})")
    try:
        hits = attach_sequences(taxid, select_top_hits(run_blastp(qpath, taxid)))
        item = {"taxid": taxid, "label": t["label"], "status": "hits" if hits else "no_hits",
                "n_hits": len(hits), "hits": hits, "completed": utc_now()}
        save_json(checkpoint_path(taxid), item)
        completed[taxid] = item
        export_results(completed)
        event("searched", taxid=taxid, query=QUERY_NAME, n_hits=len(hits))
        print(f"    {len(hits)} hit(s)" + (": " + ", ".join(h["sseqid"] for h in hits) if hits else ""))
    except Exception as exc:
        event("search_error", taxid=taxid, query=QUERY_NAME, error=f"{type(exc).__name__}: {exc}")
        print(f"    ERROR: {type(exc).__name__}: {exc}")

completed = load_checkpoints()
export_results(completed)
total_hits = sum(len(v.get("hits", [])) for v in completed.values())
print(f"\n{QUERY_NAME}: searched {len(completed)}/{len(ready)} taxa, {total_hits} hits total.")
print(f"  table:  {SUMMARY_CSV}")
print(f"  FASTA:  {HITS_FASTA}")


New/changed query or thresholds - previous checkpoints for this query cleared.
0/194 taxa already searched for Cyt-c_cyano; 194 to go.
[1/194] Aquifex aeolicus VF5 (TaxID 224324)
    0 hit(s)
[2/194] Paracoccus denitrificans (TaxID 266)
    1 hit(s): WP_011748139.1
[3/194] Escherichia coli K-12 subMG1655 (TaxID 83333)
    0 hit(s)
[4/194] Thermus thermophilus HB8 (TaxID 300852)
    1 hit(s): WP_011229026.1
[5/194] Thermus thermophilus JL-18 (TaxID 798128)
    1 hit(s): WP_014630378.1
[6/194] Rhodobacter capsulatus SB1003 (TaxID 272942)
    0 hit(s)
[7/194] Rickettsia prowazekii (TaxID 782)
    0 hit(s)
[8/194] Yarrowia lipolytica (TaxID 4952)
    0 hit(s)
[9/194] Tetrahymena thermophila (TaxID 312017)
    0 hit(s)
[10/194] Homo sapiens (TaxID 9606)
    0 hit(s)
[11/194] Bos taurus (TaxID 9913)
    0 hit(s)
[12/194] Drosophila melanogaster (TaxID 7227)
    0 hit(s)
[13/194] Mus musculus (TaxID 10090)
    0 hit(s)
[14/194] Ovis aries (TaxID 9940)
    0 hit(s)
[15/194] Acaryochloris marin

## 6. Re-running, parameter sweeps, and other proteins

- **Another protein:** drop its FASTA in `queries/` (if not already there), set `QUERY_FASTA` to that
  filename in Section 1, and Run all. `QUERY_NAME` auto-names from the file unless you set it.
- **Parameter sweep on the same query:** change `EVALUE` / `TOP_N` / `MIN_QUERY_COVERAGE` **and** give
  `RUN_LABEL` a new name, then Run all. Results go to a fresh
  `Outputs/<COLLECTION>/<QUERY_NAME>/<RUN_LABEL>/` folder; the previous run is preserved. (Same
  `RUN_LABEL` with changed parameters -> Section 2 warns you and the search replaces that folder.)
- **Re-search one organism:** delete its `state/<taxid>.result.json` inside the run's results folder.
- **Grow or rebuild the library:** that is the Proteome Library Builder's job, not this notebook.

Outputs: `Outputs/<COLLECTION>/<QUERY_NAME>/<RUN_LABEL>/` — `*_summary.csv`, `*_hits.faa`, `events.jsonl`.